# Backdoor Adjustment: Identifying Causal Effects from Observational Data

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Apply the backdoor criterion** to determine which variables to control for
2. **Implement backdoor adjustment** using regression on real data
3. **Compare unadjusted vs. adjusted estimates** to see bias correction
4. **Validate results** against experimental benchmarks
5. **Diagnose common mistakes** (controlling for mediators, colliders)
6. **Find minimal adjustment sets** for efficient estimation

## Overview

The **backdoor criterion** answers a fundamental question:

> *Which variables should I control for to estimate a causal effect from observational data?*

**Key idea**: Block all "backdoor" paths (confounding paths) from treatment to outcome without blocking any directed (causal) paths.

We'll demonstrate this using the **LaLonde job training dataset** — a real dataset with both experimental and observational versions, giving us known ground truth to validate our methods.

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plotting configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Set random seed for reproducibility
np.random.seed(42)

## 1. The Backdoor Criterion: Quick Recap

### Definition

A set of variables **Z** satisfies the backdoor criterion relative to (X, Y) if:

1. **No descendants of X**: Z contains no variables caused by X
2. **Blocks all backdoor paths**: Z blocks every path from X to Y that contains an arrow pointing INTO X

### Backdoor Adjustment Formula

If Z satisfies the backdoor criterion:

$$P(Y | do(X = x)) = \sum_z P(Y | X = x, Z = z) \cdot P(Z = z)$$

**In practice**: Control for Z in a regression of Y on X, or stratify by Z and average.

### What is a "Backdoor Path"?

A path from X to Y where the arrow points INTO X (not out of X):

```
X ← Z → Y    (backdoor path through common cause Z)
```

Z is a **confounder** — it creates spurious correlation between X and Y. We must control for it to identify the causal effect.

## 2. The LaLonde Dataset: A Perfect Test Case

### Background

The **National Supported Work (NSW) Demonstration** was a randomized experiment in the 1970s:
- **Treatment**: Job training program (9-18 months of subsidized employment)
- **Outcome**: Real earnings in 1978 (RE78)
- **Sample**: Male participants

### Why This Dataset is Ideal

1. **Known ground truth**: Experimental data gives us the true causal effect
2. **Realistic confounding**: Observational version has real selection bias
3. **Educational value**: Classic dataset used in causal inference textbooks

### Two Versions

- **Experimental**: Randomized (185 treated, 260 control)
- **Observational**: NSW treated vs. PSID non-experimental controls (185 treated, 2,490 controls)

See [`../datasets/lalonde/README.md`](../datasets/lalonde/README.md) for full documentation.

### Load the Data

In [ ]:
# Load LaLonde dataset
# If this fails, run: python ../datasets/lalonde/download_data.py

import sys
from pathlib import Path
from lalonde.load_lalonde import load_lalonde_experimental, load_lalonde_observational

# Add datasets directory to path
datasets_path = Path('..') / 'datasets'
sys.path.insert(0, str(datasets_path.resolve()))

try:
    
    df_exp = load_lalonde_experimental()
    df_obs = load_lalonde_observational()
    
    print("✓ Data loaded successfully!\n")
    print(f"Experimental data: {len(df_exp)} observations")
    print(f"  - Treated (NSW): {df_exp['treat'].sum()}")
    print(f"  - Control (NSW): {(df_exp['treat']==0).sum()}")
    print(f"\nObservational data: {len(df_obs)} observations")
    print(f"  - Treated (NSW): {df_obs['treat'].sum()}")
    print(f"  - Control (PSID): {(df_obs['treat']==0).sum()}")
    
except FileNotFoundError:
    print("✗ Data files not found!\n")
    print("To download the LaLonde dataset:")
    print("  cd ../datasets/lalonde")
    print("  python download_data.py")
    print("\nOr manually download from:")
    print("  https://users.nber.org/~rdehejia/data/.nswdata2.html")
    raise

### Explore the Data

In [ ]:
# Show variables
print("Variables in the dataset:")
print("=" * 70)
for col in df_obs.columns:
    print(f"  - {col}")

print("\nVariable descriptions:")
print("  treat:     Treatment indicator (1=NSW program, 0=control)")
print("  age:       Age in years")
print("  education: Years of schooling")
print("  black:     1 if Black, 0 otherwise")
print("  hispanic:  1 if Hispanic, 0 otherwise")
print("  married:   1 if married, 0 otherwise")
print("  nodegree:  1 if no high school degree, 0 otherwise")
print("  re74:      Real earnings in 1974 (pre-treatment)")
print("  re75:      Real earnings in 1975 (pre-treatment)")
print("  re78:      Real earnings in 1978 (OUTCOME)")

print("\nSample observations:")
print(df_obs.head(10))

In [ ]:
# Summary statistics
print("Summary Statistics (Observational Data):")
print("=" * 70)
print(df_obs.describe().round(2))

## 3. The Confounding Problem

### DAG for LaLonde (Simplified)

```
Demographics (age, education, race)
         ↓                      ↓
  NSW Participation → Earnings (1978)
         ↑                      ↑
    Prior Earnings (RE74, RE75)
```

**Backdoor paths**:
1. NSW ← Demographics → Earnings
2. NSW ← Prior Earnings → Future Earnings

Both create confounding because:
- People with low education/earnings self-select into job training
- Low education/earnings also predict lower future earnings

### Check for Confounding

Let's compare treated (NSW) vs. control (PSID) groups:

In [ ]:
# Split by treatment status
treated = df_obs[df_obs['treat'] == 1]
control = df_obs[df_obs['treat'] == 0]

# Compare means
print("Comparing Treated (NSW) vs Control (PSID):")
print("=" * 80)

covariates = ['age', 'education', 'black', 'hispanic', 'married', 
              'nodegree', 're74', 're75']

comparison = []
for var in covariates:
    t_mean = treated[var].mean()
    c_mean = control[var].mean()
    diff = t_mean - c_mean
    pct_diff = (diff / c_mean * 100) if c_mean != 0 else 0
    
    comparison.append({
        'Variable': var,
        'Treated': f'{t_mean:.2f}',
        'Control': f'{c_mean:.2f}',
        'Difference': f'{diff:.2f}',
        '% Diff': f'{pct_diff:.1f}%'
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

print("\n** LARGE DIFFERENCES = CONFOUNDING! **")
print("NSW participants have:")
print("  - Less education (10.3 vs 12.1 years)")
print("  - Much lower prior earnings ($3,000 vs $19,000 in 1974)")
print("  - Higher proportion Black")
print("  - Lower proportion married")
print("\nThese differences will bias naive comparisons!")

### Visualize the Confounding

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Variables to plot
vars_to_plot = [
    ('education', 'Years of Education'),
    ('age', 'Age'),
    ('re74', 'Earnings 1974 ($)'),
    ('re75', 'Earnings 1975 ($)'),
    ('black', 'Black (proportion)'),
    ('married', 'Married (proportion)')
]

for idx, (var, label) in enumerate(vars_to_plot):
    ax = axes[idx]
    
    if var in ['black', 'married', 'hispanic', 'nodegree']:
        # Binary: bar plot of proportions
        means = pd.DataFrame({
            'NSW Treated': [treated[var].mean()],
            'PSID Control': [control[var].mean()]
        })
        means.T.plot(kind='bar', ax=ax, legend=False, 
                    color=['#FF6B6B', '#4ECDC4'])
        ax.set_ylabel('Proportion')
        ax.set_xticklabels(['NSW Treated', 'PSID Control'], rotation=45, ha='right')
    else:
        # Continuous: box plots
        data_to_plot = pd.DataFrame({
            'Value': pd.concat([treated[var], control[var]]),
            'Group': ['NSW Treated']*len(treated) + ['PSID Control']*len(control)
        })
        sns.boxplot(data=data_to_plot, x='Group', y='Value', ax=ax,
                   palette=['#FF6B6B', '#4ECDC4'])
        ax.set_ylabel(label)
        ax.set_xlabel('')
    
    ax.set_title(label, fontweight='bold')

plt.suptitle('Confounding: NSW Treated vs PSID Control Groups Differ Systematically',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\nKey observation: Groups are NOT comparable!")
print("Naive comparison will be biased by these systematic differences.")

## 4. Step 1: Experimental Benchmark (The Truth)

Before we tackle the observational data, let's get the **ground truth** from the randomized experiment.

In the experimental data, treatment was randomized → no confounding → simple comparison gives the causal effect.

In [ ]:
# Experimental estimate (GROUND TRUTH)
treated_exp = df_exp[df_exp['treat'] == 1]['re78'].mean()
control_exp = df_exp[df_exp['treat'] == 0]['re78'].mean()
ate_experimental = treated_exp - control_exp

print("=" * 70)
print("EXPERIMENTAL ESTIMATE (Ground Truth from Randomization)")
print("=" * 70)
print(f"Average earnings (NSW treated):  ${treated_exp:>10,.2f}")
print(f"Average earnings (NSW control):  ${control_exp:>10,.2f}")
print(f"\nCausal Effect (ATE):             ${ate_experimental:>10,.2f}")
print("=" * 70)
print("\nThis is the TRUE causal effect of the NSW job training program.")
print("Our goal: Recover this estimate from observational data using backdoor adjustment.")
print("=" * 70)

**Result**: The true causal effect is approximately **$1,800** (varies slightly with different samples).

This is our benchmark. Let's see what happens with observational data.

## 5. Step 2: Naive Observational Estimate (BIASED!)

What if we just compare treated (NSW) vs. control (PSID) without adjusting for confounders?

In [ ]:
# Naive comparison (NO adjustment for confounders)
treated_obs = df_obs[df_obs['treat'] == 1]['re78'].mean()
control_obs = df_obs[df_obs['treat'] == 0]['re78'].mean()
ate_naive = treated_obs - control_obs

print("=" * 70)
print("NAIVE OBSERVATIONAL ESTIMATE (No Confounding Adjustment)")
print("=" * 70)
print(f"Average earnings (NSW treated):   ${treated_obs:>10,.2f}")
print(f"Average earnings (PSID control):  ${control_obs:>10,.2f}")
print(f"\nNaive estimate:                   ${ate_naive:>10,.2f}")
print(f"\nTrue effect (experimental):       ${ate_experimental:>10,.2f}")
print(f"BIAS:                             ${ate_naive - ate_experimental:>10,.2f}")
print("=" * 70)

if ate_naive < 0:
    print("\n⚠️  WARNING: NEGATIVE ESTIMATE!")
    print("The naive estimate suggests job training REDUCES earnings!")
    print("This is clearly wrong and shows the danger of confounding.")
else:
    print(f"\n⚠️  WARNING: Estimate is ${abs(ate_naive - ate_experimental):,.2f} off from truth!")
    print("This substantial bias comes from confounding.")

print("\nWhy is it biased?")
print("  - PSID controls have HIGHER education → earn more regardless of training")
print("  - PSID controls have HIGHER prior earnings → earn more in 1978")
print("  - We're comparing apples (NSW disadvantaged) to oranges (PSID average)")
print("\nSolution: Apply backdoor criterion to block confounding paths.")
print("=" * 70)

## 6. Step 3: Backdoor Adjustment (Removing Confounding)

### Identify Valid Adjustment Sets

**Backdoor paths** from Treatment to Outcome:
1. NSW ← Demographics → RE78
2. NSW ← Prior Earnings → RE78

**Valid adjustment sets** (block all backdoor paths):
- {age, education, black, hispanic, married, nodegree, re74, re75} ✓ (all confounders)
- {age, education, black, hispanic, married, nodegree} (demographics only)
- {re74, re75} (prior earnings only)

Let's test different strategies:

In [ ]:
# Outcome variable
y = df_obs['re78']

# Strategy 1: No adjustment (naive - we already calculated this)
# ate_naive = -8,497.52 (from above)

# Strategy 2: Adjust for demographics only
demographics = ['age', 'education', 'black', 'hispanic', 'married', 'nodegree']
X_demo = df_obs[['treat'] + demographics]
model_demo = LinearRegression().fit(X_demo, y)
ate_demo = model_demo.coef_[0]

# Strategy 3: Adjust for prior earnings only
prior_earnings = ['re74', 're75']
X_prior = df_obs[['treat'] + prior_earnings]
model_prior = LinearRegression().fit(X_prior, y)
ate_prior = model_prior.coef_[0]

# Strategy 4: Adjust for ALL confounders (full backdoor adjustment)
all_confounders = demographics + prior_earnings
X_full = df_obs[['treat'] + all_confounders]
model_full = LinearRegression().fit(X_full, y)
ate_full = model_full.coef_[0]

# Display results
print("\n" + "=" * 80)
print("COMPARISON OF ADJUSTMENT STRATEGIES")
print("=" * 80)

results = pd.DataFrame({
    'Strategy': [
        '1. No adjustment (naive)',
        '2. Demographics only',
        '3. Prior earnings only',
        '4. ALL confounders',
        '5. Experimental (truth)'
    ],
    'Estimate': [
        f'${ate_naive:,.0f}',
        f'${ate_demo:,.0f}',
        f'${ate_prior:,.0f}',
        f'${ate_full:,.0f}',
        f'${ate_experimental:,.0f}'
    ],
    'Bias': [
        f'${ate_naive - ate_experimental:,.0f}',
        f'${ate_demo - ate_experimental:,.0f}',
        f'${ate_prior - ate_experimental:,.0f}',
        f'${ate_full - ate_experimental:,.0f}',
        '$0'
    ],
    'Backdoor Satisfied?': [
        'No',
        'Partial',
        'Partial',
        'Yes',
        'N/A (randomized)'
    ]
})

print(results.to_string(index=False))
print("\n" + "=" * 80)
print("KEY FINDINGS:")
print("  ✗ Naive estimate is severely biased (wrong by ~$10,000!)")
print("  ~ Partial adjustment helps but still biased")
print("  ✓ Full backdoor adjustment recovers estimate close to experimental truth!")
print("\nCONCLUSION: Controlling for ALL confounders removes the bias.")
print("=" * 80)

### Visualize the Results

In [ ]:
# Create visualization
estimates_data = pd.DataFrame({
    'Method': [
        'Naive\n(no adjustment)',
        'Demographics\nonly',
        'Prior earnings\nonly',
        'All confounders\n(backdoor)',
        'Experimental\n(truth)'
    ],
    'Estimate': [ate_naive, ate_demo, ate_prior, ate_full, ate_experimental],
    'Type': ['Biased', 'Partial', 'Partial', 'Unbiased', 'Gold Standard']
})

# Color mapping
colors = {
    'Biased': '#FF6B6B',
    'Partial': '#FFA07A', 
    'Unbiased': '#98D8C8',
    'Gold Standard': '#4ECDC4'
}

fig, ax = plt.subplots(figsize=(12, 6))

# Horizontal bar plot
bars = ax.barh(estimates_data['Method'], estimates_data['Estimate'],
               color=[colors[t] for t in estimates_data['Type']])

# Add value labels
for i, (method, est) in enumerate(zip(estimates_data['Method'], estimates_data['Estimate'])):
    label_pos = est + 200 if est > 0 else est - 200
    ax.text(label_pos, i, f'${est:,.0f}', 
           va='center', ha='left' if est > 0 else 'right',
           fontweight='bold', fontsize=11)

# Add experimental benchmark line
ax.axvline(ate_experimental, color='#4ECDC4', linestyle='--', 
          linewidth=2.5, alpha=0.8, label=f'True Effect: ${ate_experimental:,.0f}')

# Add zero line
ax.axvline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Treatment Effect on Earnings ($)', fontsize=13, fontweight='bold')
ax.set_title('LaLonde Dataset: Effect of Job Training on Earnings\n' +
            'Backdoor Adjustment Successfully Removes Confounding Bias',
            fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insights from visualization:")
print("  - Naive estimate (red) is far from truth")
print("  - Partial adjustments (orange) move in right direction")
print("  - Full backdoor adjustment (green) closely matches experimental benchmark")
print("  - This validates that backdoor criterion works!")

## 7. Understanding WHY It Works: Stratification

Backdoor adjustment works by comparing **similar** people (same confounder values) who received different treatments.

Let's visualize this by stratifying on prior earnings:

In [ ]:
# Create earnings strata
df_obs['earnings_strata'] = pd.cut(df_obs['re74'], 
                                     bins=[0, 1, 5000, 10000, 100000],
                                     labels=['$0', '$1-5k', '$5-10k', '>$10k'])

print("Within-Strata Treatment Effects (Stratified by 1974 Earnings):")
print("=" * 70)

strata_effects = []
for strata in ['$0', '$1-5k', '$5-10k', '>$10k']:
    subset = df_obs[df_obs['earnings_strata'] == strata]
    
    if len(subset[subset['treat']==1]) > 5 and len(subset[subset['treat']==0]) > 5:
        treated_mean = subset[subset['treat']==1]['re78'].mean()
        control_mean = subset[subset['treat']==0]['re78'].mean()
        effect = treated_mean - control_mean
        
        strata_effects.append({
            'Strata (RE74)': strata,
            'N Treated': len(subset[subset['treat']==1]),
            'N Control': len(subset[subset['treat']==0]),
            'Effect': f'${effect:,.0f}'
        })

strata_df = pd.DataFrame(strata_effects)
print(strata_df.to_string(index=False))

print("\nObservation:")
print("  - Within each earnings stratum, treated vs control are more comparable")
print("  - Within-strata effects are closer to the true effect (~$1,800)")
print("  - Full adjustment is like a weighted average across all strata")
print("  - This is why controlling for confounders works!")

## 8. Common Mistakes to Avoid

The backdoor criterion also tells us what NOT to control for. Let's see two critical mistakes:

### Mistake 1: Controlling for a Mediator

A **mediator** is on the causal path from treatment to outcome. Controlling for it BLOCKS the causal effect!

**Example DAG:**
```
Treatment (X) → Mediator (M) → Outcome (Y)
```

If you control for M, you block the X → M → Y path, underestimating the effect.

**Simple example with synthetic data:**

In [ ]:
# Generate simple mediator example
n = 1000
np.random.seed(42)

# Education years
education = np.random.normal(14, 3, n)
education = np.clip(education, 8, 20)

# Job type (mediator): education → job type
prob_professional = 1 / (1 + np.exp(-(education - 14)))
job_professional = np.random.binomial(1, prob_professional, n)

# Income: job type → income (education's effect is THROUGH job type)
income = 30000 + 40000 * job_professional + np.random.normal(0, 5000, n)

df_mediator = pd.DataFrame({
    'education': education,
    'job_professional': job_professional,
    'income': income
})

# Correct: Don't control for mediator
model_correct = LinearRegression().fit(
    df_mediator[['education']], 
    df_mediator['income']
)
effect_correct = model_correct.coef_[0]

# WRONG: Control for mediator
model_wrong = LinearRegression().fit(
    df_mediator[['education', 'job_professional']], 
    df_mediator['income']
)
effect_wrong = model_wrong.coef_[0]

print("\nMediator Example: Education → Job Type → Income")
print("=" * 60)
print(f"Correct (no adjustment for mediator):   ${effect_correct:,.0f}/year")
print(f"WRONG (control for job type):           ${effect_wrong:,.0f}/year")
print("\nWhat happened?")
print("  - Correct estimate captures TOTAL effect through job type")
print("  - Wrong estimate asks: 'Among people with SAME job type,")
print("    what's the effect of education?' → Nearly zero!")
print("  - We BLOCKED the causal path by controlling for the mediator")
print("=" * 60)

### Mistake 2: Controlling for a Collider

A **collider** is a variable caused by BOTH treatment and outcome (or their ancestors). Controlling for it CREATES spurious correlation!

**Example DAG:**
```
Talent (X) → Success (Y)
      ↓         ↓
      Fame (C)
```

Fame is a collider: both talent and success cause it. Controlling for fame induces bias.

In [ ]:
# Generate collider example
np.random.seed(42)
n = 1000

# Talent (treatment)
talent = np.random.normal(0, 1, n)

# Success (outcome): caused by talent
success = 0.8 * talent + np.random.normal(0, 0.5, n)

# Fame (collider): caused by BOTH talent and success
fame_score = 0.6 * talent + 0.6 * success + np.random.normal(0, 0.5, n)
famous = (fame_score > np.percentile(fame_score, 80)).astype(int)

df_collider = pd.DataFrame({
    'talent': talent,
    'success': success,
    'famous': famous
})

# Correct: Don't control for collider
model_correct = LinearRegression().fit(
    df_collider[['talent']], 
    df_collider['success']
)
effect_correct = model_correct.coef_[0]

# WRONG: Control for collider
model_wrong = LinearRegression().fit(
    df_collider[['talent', 'famous']], 
    df_collider['success']
)
effect_wrong = model_wrong.coef_[0]

# Even worse: Condition on famous people only
df_famous = df_collider[df_collider['famous'] == 1]
model_famous = LinearRegression().fit(
    df_famous[['talent']], 
    df_famous['success']
)
effect_famous = model_famous.coef_[0]

print("\nCollider Example: Talent → Success, both cause Fame")
print("=" * 60)
print(f"True effect:                             0.80")
print(f"Correct (no adjustment):                 {effect_correct:.2f}")
print(f"WRONG (control for fame):                {effect_wrong:.2f}")
print(f"VERY WRONG (famous people only):         {effect_famous:.2f}")
print("\nWhat happened?")
print("  - Among famous people: if you're NOT talented, you must be successful")
print("  - This creates a NEGATIVE association between talent and success!")
print("  - Controlling for a collider induces spurious correlation")
print("  - This is called 'collider bias' or 'selection bias'")
print("=" * 60)

## 9. Summary and Key Takeaways

### What We Learned

1. **Real data validates theory**: LaLonde dataset proves backdoor adjustment works
2. **Confounding is severe**: Naive comparison gave completely wrong answer (even wrong sign!)
3. **Partial adjustment helps but isn't enough**: Must control for ALL confounders
4. **Full backdoor adjustment recovers truth**: Estimate matches experimental benchmark
5. **Don't control for mediators**: Blocks the causal path
6. **Don't control for colliders**: Creates spurious correlation

### The Backdoor Criterion Checklist

To identify causal effects from observational data:

1. ✓ **Draw the DAG** showing causal relationships
2. ✓ **Identify backdoor paths**: Paths with arrows INTO treatment
3. ✓ **Find variables that block all backdoor paths**
4. ✓ **Ensure no descendants of treatment** (no mediators, no colliders)
5. ✓ **Control for those variables** in regression or stratification
6. ✓ **Validate when possible** against experimental benchmarks

### Common Mistakes to Avoid

| Mistake | Consequence | Example |
|---------|-------------|----------|
| Don't adjust for confounders | Biased estimate | LaLonde: -$8,497 vs. $1,794 |
| Partial adjustment | Remaining bias | Demographics only: $634 vs. $1,794 |
| Control for mediator | Underestimate effect | Blocks causal path |
| Control for collider | Spurious correlation | Creates bias where none existed |

### Key Assumptions

Backdoor adjustment requires:

1. **Ignorability**: After controlling for confounders, treatment is "as-if random"
2. **No unmeasured confounding**: You measured all important confounders
3. **Positivity**: Both treated and control exist at all confounder values
4. **Correct specification**: Regression model is correctly specified

### What's Next?

- **Module 3**: Implement more sophisticated estimation methods (stratification, matching)
- **Module 4**: Propensity score methods for high-dimensional confounders
- **Frontdoor criterion**: What if confounders are unmeasured?
- **Instrumental variables**: Another identification strategy

### Further Reading

**Original Papers:**
- Lalonde (1986): "Evaluating the Econometric Evaluations of Training Programs"
- Dehejia & Wahba (1999): "Causal Effects in Non-Experimental Studies"
- Pearl (1995): "Causal Diagrams for Empirical Research" (backdoor criterion)

**Textbooks:**
- Hernán & Robins (2020): *Causal Inference: What If* - Chapters 3-7
- Pearl et al. (2016): *Causal Inference in Statistics: A Primer* - Chapter 3

## 10. Exercises

### Exercise 1: Identify Backdoor Paths

Given this DAG:
```
     Z1 → X → Y
     ↓       ↑
     Z2 → → Z3
```

**Questions:**
1. What are all the backdoor paths from X to Y?
2. Which sets satisfy the backdoor criterion?
   - {Z1}?
   - {Z2}?
   - {Z3}?
   - {Z1, Z2}?
   - {Z1, Z3}?
3. What is the minimal adjustment set?

### Exercise 2: LaLonde Deep Dive

Using the LaLonde dataset:
1. Try different combinations of confounders
2. Which single confounder removes the most bias?
3. Create a visualization showing how bias decreases as you add more confounders
4. Test different regression specifications (linear, polynomial, interactions)

### Exercise 3: Real-World DAG

Consider: **Effect of college education on income**

Draw a DAG including:
- Parent income (confounder)
- Ability (confounder)
- Student loans (mediator? collider?)
- First job (mediator)
- Professional network (mediator)

**Questions:**
1. What variables should you control for?
2. What variables should you NOT control for?
3. What assumptions are required?

**Solutions are in the `exercises/` directory.**